# LayerNorm/RMSNorm 학습 안정성 실습 - 추가 실습 코드: LayerNorm vs BatchNorm 비교

- Tutorial ID: `mod-layernorm-rmsnorm-lab`
- Tutorial: LayerNorm/RMSNorm 학습 안정성 실습
- Section ID: `practice-layernorm-vs-batchnorm`
- Section: 추가 실습 코드: LayerNorm vs BatchNorm 비교


In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: LayerNorm vs BatchNorm 비교
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def batch_norm(x, gamma=1.0, beta=0.0, eps=1e-5):
    """
    Batch Normalization
    - 배치 차원에서 평균/분산 계산
    - x: (batch_size, features)
    """
    mean = np.mean(x, axis=0)  # 배치에 대해 평균
    var = np.var(x, axis=0)
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta, mean, var

def layer_norm(x, gamma=1.0, beta=0.0, eps=1e-5):
    """
    Layer Normalization
    - 특성 차원에서 평균/분산 계산
    - x: (batch_size, features)
    """
    mean = np.mean(x, axis=-1, keepdims=True)  # 특성에 대해 평균
    var = np.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta, mean.flatten(), var.flatten()

def rms_norm(x, gamma=1.0, eps=1e-5):
    """
    RMS Normalization (LLaMA, Mistral에서 사용)
    - 평균 빼기 없이 RMS로만 정규화
    """
    rms = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + eps)
    return gamma * x / rms, rms.flatten()

# 테스트 데이터
np.random.seed(42)
batch_size = 4
features = 8

# 다양한 스케일의 입력
x = np.random.randn(batch_size, features) * np.array([1, 10, 0.1, 5, 2, 0.5, 8, 3])

print("=" * 60)
print("Normalization 기법 비교")
print("=" * 60)

print(f"\\n입력 shape: {x.shape} (batch={batch_size}, features={features})")
print(f"입력 통계:")
print(f"  전체 평균: {x.mean():.3f}, 전체 표준편차: {x.std():.3f}")

print("\\n━━━ Batch Normalization ━━━")
print("정규화 축: 배치 (각 특성별로 평균/분산)")
bn_out, bn_mean, bn_var = batch_norm(x)
print(f"특성별 평균: {bn_mean.round(3)}")
print(f"출력 통계 (특성별):")
print(f"  출력 평균: {bn_out.mean(axis=0).round(3)}")  # 각 특성의 평균 ≈ 0
print(f"  출력 표준편차: {bn_out.std(axis=0).round(3)}")  # 각 특성의 std ≈ 1

print("\\n━━━ Layer Normalization ━━━")
print("정규화 축: 특성 (각 샘플별로 평균/분산)")
ln_out, ln_mean, ln_var = layer_norm(x)
print(f"샘플별 평균: {ln_mean.round(3)}")
print(f"출력 통계 (샘플별):")
print(f"  출력 평균: {ln_out.mean(axis=1).round(3)}")  # 각 샘플의 평균 ≈ 0
print(f"  출력 표준편차: {ln_out.std(axis=1).round(3)}")  # 각 샘플의 std ≈ 1

print("\\n━━━ RMS Normalization ━━━")
print("정규화 축: 특성 (평균 빼기 없이 RMS만)")
rms_out, rms_vals = rms_norm(x)
print(f"샘플별 RMS: {rms_vals.round(3)}")
print(f"출력 RMS: {np.sqrt(np.mean(rms_out**2, axis=1)).round(3)}")  # ≈ 1

print("\\n📊 NLP에서 LayerNorm을 선호하는 이유:")
print("  1. 시퀀스 길이가 가변적 (BN은 배치 통계 문제)")
print("  2. 작은 배치에서도 안정적")
print("  3. 추론 시 배치 통계 불필요")